In [2]:
import gymnasium as gym
from stable_baselines3 import PPO
from stable_baselines3.common.evaluation import evaluate_policy

# 1. 创建环境（开启 render_mode='human' 可以看到动画）
env = gym.make("CartPole-v1", render_mode="human")

# 2. 创建 PPO 智能体（使用多层感知机 MLP 策略）
model = PPO("MlpPolicy", env, verbose=1)

# 3. 训练模型（总步数 10,000 步）
print("开始训练...")
model.learn(total_timesteps=10_000)
print("训练完成！")

# 4. 保存模型（可选）
model.save("ppo_cartpole")

# 5. 测试训练好的智能体
print("正在演示智能体...")
obs, info = env.reset()
for i in range(1000):
    action, _states = model.predict(obs, deterministic=True)
    obs, reward, terminated, truncated, info = env.step(action)
    env.render()  # 显示画面
    if terminated or truncated:
        obs, info = env.reset()

env.close()

Using cpu device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.
开始训练...


D:\pycharm jupyter git\.venv\Lib\site-packages\pygame\pkgdata.py:25: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream, resource_exists


---------------------------------
| rollout/           |          |
|    ep_len_mean     | 21.7     |
|    ep_rew_mean     | 21.7     |
| time/              |          |
|    fps             | 45       |
|    iterations      | 1        |
|    time_elapsed    | 44       |
|    total_timesteps | 2048     |
---------------------------------
-----------------------------------------
| rollout/                |             |
|    ep_len_mean          | 26.1        |
|    ep_rew_mean          | 26.1        |
| time/                   |             |
|    fps                  | 45          |
|    iterations           | 2           |
|    time_elapsed         | 90          |
|    total_timesteps      | 4096        |
| train/                  |             |
|    approx_kl            | 0.009644598 |
|    clip_fraction        | 0.118       |
|    clip_range           | 0.2         |
|    entropy_loss         | -0.686      |
|    explained_variance   | 0.000472    |
|    learning_rate        | 0.

In [ ]:
# ==============================
# 强化学习入门：训练 AI 玩 Flappy Bird！
# 使用 flappy-bird-gymnasium + Stable-Baselines3
# ==============================

import gymnasium as gym
from stable_baselines3 import PPO
from stable_baselines3.common.evaluation import evaluate_policy
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.callbacks import EvalCallback, StopTrainingOnRewardThreshold
import os
import numpy as np

# ==============================
# 第一步：创建环境
# ==============================
# 数学意义：定义马尔可夫决策过程(MDP) - 状态空间、动作空间、奖励函数
# 编程意义：初始化游戏环境，提供与游戏交互的接口
# 替换可能：可替换为其他gym环境如"CartPole-v1", "LunarLander-v2"等
print("🎮 创建 Flappy Bird 环境...")

# 创建训练环境（无渲染以提高训练速度）
# render_mode=None: 训练时不显示画面，加快训练速度
# use_lidar=False: 使用常规观测空间（更简单）
train_env = gym.make("FlappyBird-v0", render_mode=None, use_lidar=False)

# 创建评估环境（有渲染用于演示）
eval_env = gym.make("FlappyBird-v0", render_mode="human", use_lidar=False)

print(f"✅ 环境创建成功！")
print(f"   动作空间: {train_env.action_space}")  # 离散动作：0=不飞，1=飞
print(f"   观测空间: {train_env.observation_space}")  # 状态观测维度

# ==============================
# 第二步：创建智能体（PPO 算法）
# ==============================
# 数学意义：近端策略优化(PPO) - 策略梯度方法，通过裁剪保证策略更新稳定性
# 编程意义：创建PPO模型，包含策略网络和价值网络
# 替换可能：可替换为其他算法如A2C, DQN, SAC等
print("\n🤖 创建 PPO 智能体...")

# 参数说明：
# "MlpPolicy": 使用多层感知机作为策略网络
# env: 训练环境
# learning_rate: 学习率，控制参数更新步长
# n_steps: 每次更新前收集的步数
# batch_size: 小批量大小
# verbose=1: 显示训练信息
model = PPO(
    "MlpPolicy",
    train_env,
    learning_rate=1e-3,
    n_steps=2048,           # 每次更新前收集的步数
    batch_size=64,          # 小批量大小
    n_epochs=10,            # 每次更新时优化epoch数
    gamma=0.99,             # 折扣因子，权衡当前与未来奖励
    gae_lambda=0.95,        # GAE参数，权衡偏差和方差
    clip_range=0.2,         # PPO裁剪参数，保证策略更新稳定
    verbose=1,
    tensorboard_log="./tensorboard_logs/"  # TensorBoard日志目录
)

print("✅ PPO 智能体创建成功！")

# ==============================
# 可选：设置回调函数用于早停和评估
# ==============================
# 数学意义：在训练过程中监控性能，达到阈值时停止训练
# 编程意义：自动保存最佳模型并在满足条件时停止训练
# 替换可能：可自定义回调函数实现更多功能
print("\n📊 设置训练回调函数...")

# 创建日志目录
os.makedirs("temp/logs/", exist_ok=True)

# 评估回调：定期评估模型并保存最佳版本
eval_callback = EvalCallback(
    eval_env,
    best_model_save_path="temp/logs/",
    log_path="temp/logs/",
    eval_freq=5000,  # 每5000步评估一次
    deterministic=True,
    render=False
)

# ==============================
# 第三步：开始训练
# ==============================
print("\n🚀 开始训练 AI 玩 Flappy Bird...")
print("   你会看到训练进度信息...")
print("   训练大约需要 2~5 分钟...")

# 数学意义：通过与环境交互收集经验，优化策略参数以最大化累积奖励
# 编程意义：执行训练循环，更新神经网络权重
# 替换可能：可调整训练步数或使用分布式训练
try:
    # 训练模型
    model.learn(
        total_timesteps=100000_000,  # 总共训练步数
        callback=eval_callback,   # 使用回调函数
        progress_bar=True         # 显示进度条
    )

    print("✅ 训练完成！")

    # 保存最终模型
    model.save("ppo_flappy_bird_final")
    print("💾 最终模型已保存为 'ppo_flappy_bird_final.zip'")

except KeyboardInterrupt:
    print("\n⚠️ 训练被用户中断！")
    model.save("ppo_flappy_bird_interrupted")
    print("💾 中断模型已保存为 'ppo_flappy_bird_interrupted.zip'")

# ==============================
# 第四步：评估训练好的模型
# ==============================
print("\n📈 评估模型性能...")

# 数学意义：定量评估学习策略的平均性能
# 编程意义：在多个episode上测试模型，计算平均奖励
# 替换可能：可自定义评估指标如成功率、平均生存时间等
try:
    # 加载最佳模型（如果有）
    if os.path.exists("temp/logs/best_model.zip"):
        best_model = PPO.load("temp/logs/best_model.zip")
        print("✅ 加载最佳模型进行评估")
    else:
        best_model = model
        print("ℹ️ 使用最终模型进行评估")

    # 评估模型
    mean_reward, std_reward = evaluate_policy(
        best_model,
        eval_env,
        n_eval_episodes=10,  # 评估10次
        deterministic=True   # 使用确定性策略
    )

    print(f"📊 评估结果:")
    print(f"   平均奖励: {mean_reward:.2f} ± {std_reward:.2f}")

except Exception as e:
    print(f"❌ 评估过程中出现错误: {e}")
    best_model = model

# ==============================
# 第五步：演示训练好的 AI
# ==============================
print("\n🎮 演示 AI 玩游戏...")

# 关闭训练环境
train_env.close()

# 创建新的演示环境
demo_env = gym.make("FlappyBird-v0", render_mode="human", use_lidar=False)

# 数学意义：展示学习到的策略在实际环境中的表现
# 编程意义：运行训练好的模型并可视化结果
# 替换可能：可录制视频或保存游戏回放
print("🕹️ 开始演示！按 Ctrl+C 停止...")

try:
    # 演示多个episode
    for episode in range(5):  # 演示5局游戏
        print(f"\n第 {episode + 1} 局开始...")

        obs, info = demo_env.reset()
        total_reward = 0
        steps = 0

        while True:
            # 使用训练好的模型选择动作
            # 数学意义：根据当前状态选择最优动作
            # 编程意义：神经网络前向传播，输出动作概率
            action, _states = best_model.predict(obs, deterministic=True)

            # 执行动作
            obs, reward, terminated, truncated, info = demo_env.step(action)

            total_reward += reward
            steps += 1

            # 检查游戏是否结束
            if terminated or truncated:
                print(f"   第 {episode + 1} 局结束: 得分 = {info.get('score', 0)}, 步数 = {steps}")
                break

            # 可选：添加延迟以便观察
            # import time
            # time.sleep(0.01)

except KeyboardInterrupt:
    print("\n⏹️ 演示被用户中断")

finally:
    # 确保环境被正确关闭
    demo_env.close()
    print("✅ 环境已关闭")

# ==============================
# 第六步：训练分析和建议
# ==============================
print("\n📋 训练分析总结:")

# 数学意义：分析学习曲线和性能指标
# 编程意义：提供训练结果的可视化和改进建议
print("""
🎯 性能改进建议:

1. 如果AI表现不佳:
   - 增加训练步数 (total_timesteps=500_000)
   - 调整学习率 (learning_rate=3e-4)
   - 使用更复杂的网络架构

2. 如果想尝试其他算法:
   - DQN: 适合离散动作空间
   - A2C: 更简单的策略梯度方法
   - SAC: 适合需要探索的环境

3. 高级技巧:
   - 使用课程学习 (从简单到难)
   - 添加专家演示
   - 使用集成学习

📊 要查看详细训练曲线:
   tensorboard --logdir ./tensorboard_logs/
""")

print("🎉 Flappy Bird 强化学习演示完成！")

🎮 创建 Flappy Bird 环境...
✅ 环境创建成功！
   动作空间: Discrete(2)
   观测空间: Box(-1.0, 1.0, (12,), float64)

🤖 创建 PPO 智能体...
Using cpu device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.
✅ PPO 智能体创建成功！

📊 设置训练回调函数...

🚀 开始训练 AI 玩 Flappy Bird...
   你会看到训练进度信息...
   训练大约需要 2~5 分钟...
Logging to ./tensorboard_logs/PPO_2
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 50       |
|    ep_rew_mean     | -7.6     |
| time/              |          |
|    fps             | 3592     |
|    iterations      | 1        |
|    time_elapsed    | 0        |
|    total_timesteps | 2048     |
---------------------------------
-----------------------------------------
| rollout/                |             |
|    ep_len_mean          | 50          |
|    ep_rew_mean          | -7.28       |
| time/                   |             |
|    fps                  | 2554        |
|    iterations           | 2           |
|    time_elapsed         | 1 

Eval num_timesteps=5000, episode_reward=2.00 +/- 0.00

Episode length: 31.00 +/- 0.00

-----------------------------------------
| eval/                   |             |
|    mean_ep_length       | 31          |
|    mean_reward          | 2           |
| time/                   |             |
|    total_timesteps      | 5000        |
| train/                  |             |
|    approx_kl            | 0.012378519 |
|    clip_fraction        | 0.142       |
|    clip_range           | 0.2         |
|    entropy_loss         | -0.662      |
|    explained_variance   | 0.781       |
|    learning_rate        | 0.001       |
|    loss                 | 0.104       |
|    n_updates            | 20          |
|    policy_gradient_loss | -0.00827    |
|    value_loss           | 0.52        |
-----------------------------------------


New best mean reward!

...

Episode length: 31.00 +/- 0.00

-----------------------------------------
| eval/                   |             |
|    mean_ep_length       | 31          |
|    mean_reward          | 2           |
| time/                   |             |
|    total_timesteps      | 340000      |
| train/                  |             |
|    approx_kl            | 0.018048935 |
|    clip_fraction        | 0.0606      |
|    clip_range           | 0.2         |
|    entropy_loss         | -0.0987     |
|    explained_variance   | 0.0914      |
|    learning_rate        | 0.001       |
|    loss                 | 0.708       |
|    n_updates            | 1660        |
|    policy_gradient_loss | 0.000725    |
|    value_loss           | 2.33        |
-----------------------------------------
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 240      |
|    ep_rew_mean     | 27.5     |
| time/              |          |
|    fps             | 121      |
|    iterations      | 167      |
|    t

Eval num_timesteps=345000, episode_reward=2.00 +/- 0.00

Episode length: 31.00 +/- 0.00

----------------------------------------
| eval/                   |            |
|    mean_ep_length       | 31         |
|    mean_reward          | 2          |
| time/                   |            |
|    total_timesteps      | 345000     |
| train/                  |            |
|    approx_kl            | 0.03231191 |
|    clip_fraction        | 0.0747     |
|    clip_range           | 0.2        |
|    entropy_loss         | -0.116     |
|    explained_variance   | 0.36       |
|    learning_rate        | 0.001      |
|    loss                 | 0.461      |
|    n_updates            | 1680       |
|    policy_gradient_loss | 0.00925    |
|    value_loss           | 1.15       |
----------------------------------------
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 248      |
|    ep_rew_mean     | 28.5     |
| time/              |          |
|    fps             | 122      |
|    iterations      | 169      |
|    time_elapsed    | 2